# Next-word predictor with an LSTM
Text: the first 48 of Aesop's fables (Project Gutenberg eBook #21, tr. G. F. Townsend), one fable per line in
data/fables.txt. Fables 1-40 train the model; fables 41-48 are held out to check it on text it has not seen.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import re
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
DATA = Path("data")
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. The text, sentence by sentence

In [ ]:
fables = DATA.joinpath("fables.txt").read_text().strip().split("\n")
train_fables, val_fables = fables[:40], fables[40:]


def sentences(texts):
    """Split every fable at . ! or ? followed by a space."""
    return [s for t in texts for s in re.split(r"(?<=[.!?])\s+", t) if len(s.split()) > 1]


train_sent, val_sent = sentences(train_fables), sentences(val_fables)
print(len(train_sent), "training sentences;", len(val_sent), "validation sentences")
print(train_sent[:3])

## 2. A number for every word (vocabulary from the training fables only)

In [ ]:
vectorizer = keras.layers.TextVectorization()       # lower-case, strip punctuation, split on spaces
vectorizer.adapt(train_sent)
vocab = vectorizer.get_vocabulary()                  # index 0 = padding '', index 1 = [UNK]
V = len(vocab)
print("vocabulary size (with '' and [UNK]):", V)
print(vocab[:12])


def to_ids(s):
    return [int(i) for i in vectorizer([s]).numpy()[0] if i != 0]


print(train_sent[0], "->", to_ids(train_sent[0]))

## 3. Input-output pairs: every prefix of a sentence predicts its next word

In [ ]:
def ngrams(sents):
    seqs = []
    for s in sents:
        ids = to_ids(s)
        for i in range(1, len(ids)):
            seqs.append(ids[:i + 1])        # first i words as input, word i+1 as output
    return seqs


train_seqs, val_seqs = ngrams(train_sent), ngrams(val_sent)
for s in train_seqs[:4]:
    print(s[:-1], "->", s[-1], "   ", " ".join(vocab[i] for i in s[:-1]), "->", vocab[s[-1]])
max_len = max(len(s) for s in train_seqs)
print("pairs:", len(train_seqs), "training,", len(val_seqs), "validation; longest sequence:", max_len)

In [ ]:
# Zeros in front, so the last word of the input is always just before the output
train_pad = keras.utils.pad_sequences(train_seqs, maxlen=max_len, padding="pre")
val_pad = keras.utils.pad_sequences(val_seqs, maxlen=max_len, padding="pre")
X, y = train_pad[:, :-1], train_pad[:, -1]
X_val, y_val = val_pad[:, :-1], val_pad[:, -1]
print("first padded row:", train_pad[0][-6:], " X shape", X.shape, " y shape", y.shape)

In [ ]:
# Classification over the vocabulary: one-hot targets, one column per word
Y = keras.utils.to_categorical(y, num_classes=V)
Y_val = keras.utils.to_categorical(y_val, num_classes=V)
print("Y shape:", Y.shape, " the first target is word", y[0], "=", repr(vocab[y[0]]))
unk_share = float(np.mean(y_val == 1))
print(f"validation targets that are unknown words: {unk_share:.3f}")

## 4. Embedding -> LSTM -> Dense softmax

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(max_len - 1,)),
    keras.layers.Embedding(V, 100),
    keras.layers.LSTM(150),
    keras.layers.Dense(V, activation="softmax")])
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()
counts = [l.count_params() for l in model.layers]
print("parameters per layer:", counts)
assert counts[1] == 4 * ((150 + 100) * 150 + 150)

In [ ]:
hist = model.fit(X, Y, epochs=100, batch_size=32, validation_data=(X_val, Y_val), verbose=0)
h = pd.DataFrame(hist.history)
h.insert(0, "epoch", np.arange(1, len(h) + 1))
h.round(4).to_csv(DATA / "history.csv", index=False)
print(h.iloc[[0, 9, 19, 49, 99]].round(3).to_string(index=False))
print("best validation accuracy:", h.val_accuracy.max().round(3), "at epoch", int(h.epoch[h.val_accuracy.idxmax()]))
print("validation accuracy if we always guessed the most common word 'the':",
      round(float(np.mean(y_val == vocab.index("the"))), 3))

## 5. Predicting the next word, then the next 10

In [ ]:
def next_word(text):
    ids = keras.utils.pad_sequences([to_ids(text)], maxlen=max_len - 1, padding="pre")
    p = model.predict(ids, verbose=0)[0]          # one probability per word of the vocabulary
    return vocab[int(np.argmax(p))], float(p.max())


def generate(text, n=10):
    for _ in range(n):
        text = text + " " + next_word(text)[0]
    return text


print(next_word("a lion was awakened from sleep by a"))
prompts = {"from the training fables": ["a lion was awakened from sleep by a", "the fox", "a wolf"],
           "new to the model": ["the old man said", "the king of the"]}
rows = []
for kind, ps in prompts.items():
    for p in ps:
        out = generate(p, 10)
        rows.append(dict(kind=kind, prompt=p, generated=out[len(p) + 1:]))
        print(f"{kind:25} | {p} -> {out[len(p) + 1:]}")
pd.DataFrame(rows).to_csv(DATA / "generated.csv", index=False)

In [ ]:
# The real continuation of the first prompt, for comparison
print(train_sent[0])